In [ ]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import torch

from transformers import DistilBertTokenizer, DistilBertModel
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import faiss

In [ ]:
# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_csv("Train.csv")

text_column = df.columns[0]
label_column = df.columns[1]

df[text_column] = df[text_column].astype(str)

In [ ]:
# ============================================================
# LOAD DISTILBERT
# ============================================================

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

In [ ]:
# ============================================================
# EMBEDDING FUNCTION
# ============================================================

def get_embedding(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # CLS token embedding
    embedding = outputs.last_hidden_state[:, 0, :]
    
    return embedding.numpy().flatten()


In [ ]:

# ============================================================
# CREATE EMBEDDINGS
# ============================================================

embeddings = []

for text in df[text_column]:
    embeddings.append(get_embedding(text))

X = np.array(embeddings).astype('float32')
y = df['label'].values


In [ ]:

# ============================================================
# FAISS INDEX
# ============================================================

dimension = X.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(X)


In [ ]:

# ============================================================
# RETRIEVE SIMILAR
# ============================================================

def retrieve(query, k=5):
    query_vec = get_embedding(query).astype('float32')
    
    D, I = index.search(np.array([query_vec]), k)
    
    return y[I[0]]


In [ ]:

# ============================================================
# PREDICT
# ============================================================

def predict(text):
    labels = retrieve(text)
    
    return np.bincount(labels).argmax()


In [ ]:
# ============================================================
# TEST SAMPLE
# ============================================================

sample = "python machine learning data science"
pred = predict(sample)

print("Predicted Class:", le.inverse_transform([pred]))


In [ ]:

# ============================================================
# STREAMLIT UI
# ============================================================

import streamlit as st

st.title("Resume Classification using RAG + DistilBERT 🚀")

user_input = st.text_area("Enter Resume Text")

if st.button("Predict"):
    result = predict(user_input)
    label = le.inverse_transform([result])
    
    st.success(f"Predicted Category: {label[0]}")
